In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as psf
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

spark: SparkSession = (
    SparkSession.builder.config(
        "spark.driver.extraJavaOptions", "-Djava.security.manager=allow"
    )
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)

In [ ]:
df_batch_measurements = spark.read.parquet("../data/batch_measurements.parquet")
df_measurements = spark.read.parquet("../data/measurements.parquet")

print("=== Batch Measurements ===")
print(df_batch_measurements.show(2))
print("=== Measurements ===")
print(df_measurements.show(2))

Was there in 2022 an injured polar bear older than 15 (i.e. a senior polar bear)?


In [ ]:
(
    df_batch_measurements.filter(psf.col("vet_health_check") == "INJURED")
    .filter(psf.col("age") > 15)
    .filter(psf.year("timestamp") == 2022)
    .select("name")
    .distinct()
).show()

In [ ]:
# Alernative: define expression

injured_senior_expr = (
    (psf.col("vet_health_check") == "INJURED")
    & (psf.col("age") > 15)
    & (psf.year("timestamp") == 2022)
)

df_batch_measurements.filter(injured_senior_expr).select("name").distinct().show()

In [ ]:
# Alternative: Clean the data first


def clean(df: DataFrame) -> DataFrame:
    return df.withColumn("name", psf.initcap(psf.col("name")))


df_batch_measurements.transform(clean).filter(injured_senior_expr).select(
    "name"
).distinct().show()

Bonus question: can you answer the same question using Polars [categorical types](https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#data-type-categorical)?


How many times was Blizzard Bob's name capitalized in the batch measurements?


In [ ]:
name = "Blizzard Bob"

(
    df_batch_measurements.filter(psf.lower(psf.col("name")) == name.lower())
    .groupBy("name")
    .agg(psf.count("name").alias("count"))
    .show()
)

Was Chilly Willy ever sick with a temperature above 40 degrees? (Hint: you might need to perform some data wrangling by performing a union and downfill.)


In [ ]:
name = "Chilly Willy"

window_spec = (
    Window.partitionBy("name")
    .orderBy("timestamp")
    .rowsBetween(Window.unboundedPreceding, 0)
)

(
    # Union
    # Combine the batch measurements, containing health checks,
    # with the real-time measurements containing sensor values
    df_batch_measurements.transform(clean)
    .unionByName(df_measurements, allowMissingColumns=True)
    .filter(psf.col("name") == name)
    # Downfill
    # Create a (sorted) timeline for all Chilly Willy events,
    # containing both health check and the temperature reading
    .orderBy("timestamp")
    .withColumn(
        "temperature", psf.last("temperature", ignorenulls=True).over(window_spec)
    )
    .withColumn(
        "vet_health_check",
        psf.last("vet_health_check", ignorenulls=True).over(window_spec),
    )
    # and Filter
    .filter((psf.col("vet_health_check") == "SICK") & (psf.col("temperature") > 40))
    .select("name", "temperature", "vet_health_check")
).show(5)

In [ ]:
# Alternative: using pyspark.pandas API
# This provides a pandas-like ffill() method

name = "Chilly Willy"

(
    # Union with allowMissingColumns
    df_batch_measurements.transform(clean)
    .unionByName(df_measurements, allowMissingColumns=True)
    .filter(psf.col("name") == name)
    .orderBy("timestamp")
    .pandas_api()
    .ffill()
    .to_spark()
    .filter((psf.col("vet_health_check") == "SICK") & (psf.col("temperature") > 40))
    .select("name", "temperature", "vet_health_check")
).show(5)